# FRS V7.1 K² — Live Classroom Demo Session on Kaggle
**Institution**: Kakatiya University College of Engineering & Technology (KUCET), Warangal  
**Target Environment**: Kaggle Notebooks (Dual Tesla T4 GPUs `GPU T4 x 2` or CPU)  
**Course Offering**: `CS401: Distributed Systems & Cloud Computing`  
**Faculty Coordinator**: Dr. V. Ramana Babu (`KUCET-FAC-CSE-001`)  
**Backend Control Plane**: Live Ngrok Tunnel Bridge (`https://straddle-deflate-footwear.ngrok-free.dev`)  
**Hugging Face Hub Registry**: `https://huggingface.co/Koushik-Katkam/kucet-frs-face-recognition`

---

### Features in this Demo Session:
1. **Newly Fine-Tuned Model Suite**: Uses the newly fine-tuned weights from Hugging Face Hub:
   - SFace Fine-Tuned (128-D, $\Delta s = 0.8867$, SHA-256 `0d980b10...`)
   - EdgeFace-Base (512-D, $\Delta s = 0.9504$, SHA-256 `ec401c2b...`)
   - GhostFaceNet (128-D, $\Delta s = 0.9796$, SHA-256 `1047dd31...`)
   - MiniFASNetV2 PAD (80x80 BGR, 100% Anti-Spoofing, SHA-256 `0961635e...`)
2. **Resolution-Adaptive Cosine Thresholding**: Dynamically adjusts required similarity based on face bounding box width:
   - Width $< 50$ px (distant crowd): $	au_{\text{effective}} = \max(\tau, 0.480)$
   - $50 \le \text{width} < 90$ px (classroom): $	au_{\text{effective}} = \max(\tau, 0.420)$
   - Width $\ge 90$ px (close portrait): $\tau$
3. **Department / Cohort Scoping Filter**: Scopes matching to `ALL`, `CSD`, or `CSE`.
4. **MiniFASNet Presentation Attack Detection (PAD)**: Independent live anti-spoofing gate.
5. **Exact Canvas Bounding Box Parity**: Letterboxed image rectangle mapping matching Flutter mobile and Next.js admin console.
6. **Cryptographic HMAC-SHA256 Commit Receipt**: Immutable ledger verification for institutional audit.


In [ ]:
# Step 1: Install core dependencies
!pip install -q pyngrok huggingface_hub onnx onnxruntime requests

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
print('[OK] Core dependencies verified and warnings filter initialized.')


In [ ]:
# Step 1.5: Hugging Face Hub Authentication
import os
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

if not HF_TOKEN:
    try:
        from getpass import getpass
        HF_TOKEN = getpass("Enter your Hugging Face write token (optional): ").strip()
    except Exception:
        HF_TOKEN = ""

if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        api = HfApi(token=HF_TOKEN)
        print(f"[OK] Hugging Face Hub Authenticated as: {api.whoami().get('name')}")
    except Exception as exc:
        print(f"[Notice] Hugging Face Hub note: {exc}")
else:
    print("[Notice] Running in public access mode.")


In [ ]:
# Step 2: Import standard libraries and verify OpenCV runtime
import dataclasses
import hashlib
import hmac
import io
import json
import os
from pathlib import Path
import time
from typing import Any, Dict, List, Optional, Tuple, Union
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests

try:
    from pyngrok import ngrok
    PYNGROK_AVAILABLE = True
except ImportError:
    PYNGROK_AVAILABLE = False

print("[OK] Runtime environment initialized.")
print(f"     OpenCV Version : {cv2.__version__}")
print(f"     NumPy Version  : {np.__version__}")
print(f"     pyngrok Ready  : {PYNGROK_AVAILABLE}")


In [ ]:
# Step 3: Hardware & Accelerator Audit (Tesla T4 x 2 / CUDA / CPU)
try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    cuda_available = False

print(f"CUDA Acceleration Available : {cuda_available}")
if cuda_available:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU Device #{i}             : {torch.cuda.get_device_name(i)} ({props.total_memory / (1024**3):.1f} GB)")
else:
    print("  Executing on CPU runtime.")


In [ ]:
# Step 4: Download Fine-Tuned Models from Hugging Face Hub with SHA-256 Verification
MODELS_CONFIG = {
    "yunet": {
        "name": "OpenCV Zoo YuNet Face Detector",
        "filename": "face_detection_yunet_2023mar.onnx",
        "sha256": "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4",
        "url": "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    },
    "sface_finetuned": {
        "name": "KUCET Fine-Tuned SFace Recognizer (128-D, Hugging Face Hub)",
        "filename": "kucet_frs_sface_finetuned.onnx",
        "sha256": "0d980b104c783bc4230b074d5b44b9bf329b6cc5d457ba92416b5d6b7000755f",
        "url": "https://huggingface.co/Koushik-Katkam/kucet-frs-face-recognition/resolve/main/kucet_frs_sface_finetuned.onnx",
        "repo_id": "Koushik-Katkam/kucet-frs-face-recognition",
    },
    "edgeface_finetuned": {
        "name": "KUCET Fine-Tuned EdgeFace-Base Recognizer (512-D, Hugging Face Hub)",
        "filename": "kucet_frs_edgeface_finetuned.onnx",
        "sha256": "ec401c2b048ce57bbbc0289a322ba7dbbbcc100f7d73fa61f1f9d027ca99f4f7",
        "url": "https://huggingface.co/Koushik-Katkam/kucet-frs-face-recognition/resolve/main/kucet_frs_edgeface_finetuned.onnx",
        "repo_id": "Koushik-Katkam/kucet-frs-face-recognition",
    },
    "ghostfacenet_finetuned": {
        "name": "KUCET Fine-Tuned GhostFaceNet Recognizer (128-D, Hugging Face Hub)",
        "filename": "kucet_frs_ghostfacenet_finetuned.onnx",
        "sha256": "1047dd31c520f2950d330771549e278976b674746b73230e10568cf41ecd2857",
        "url": "https://huggingface.co/Koushik-Katkam/kucet-frs-face-recognition/resolve/main/kucet_frs_ghostfacenet_finetuned.onnx",
        "repo_id": "Koushik-Katkam/kucet-frs-face-recognition",
    },
    "minifasnet_finetuned": {
        "name": "KUCET Fine-Tuned MiniFASNet Anti-Spoofing PAD (Hugging Face Hub)",
        "filename": "kucet_frs_minifasnet_finetuned.onnx",
        "sha256": "0961635e1ab4e1859bb5e7d174327f0ed5d33822b5af9539bd6fabe8057b5a2c",
        "url": "https://huggingface.co/Koushik-Katkam/kucet-frs-face-recognition/resolve/main/kucet_frs_minifasnet_finetuned.onnx",
        "repo_id": "Koushik-Katkam/kucet-frs-face-recognition",
    }
}

models_dir = Path("./models")
models_dir.mkdir(parents=True, exist_ok=True)

def verify_file_sha256(file_path: Path, expected_sha256: str) -> bool:
    if not file_path.exists():
        return False
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest().lower() == expected_sha256.lower()

def download_model(key: str, conf: dict) -> Path:
    target_path = models_dir / conf["filename"]
    expected_sha = conf.get("sha256")
    if target_path.exists() and (not expected_sha or verify_file_sha256(target_path, expected_sha)):
        print(f"[OK] Cached and verified: {conf['name']}")
        return target_path

    print(f"Downloading {conf['name']}...")
    
    # Strategy 1: Use huggingface_hub official downloader if repo_id provided
    if conf.get("repo_id"):
        try:
            from huggingface_hub import hf_hub_download
            import shutil
            downloaded = hf_hub_download(
                repo_id=conf["repo_id"],
                filename=conf["filename"],
                token=HF_TOKEN if HF_TOKEN else None
            )
            shutil.copyfile(downloaded, target_path)
            if expected_sha and not verify_file_sha256(target_path, expected_sha):
                raise ValueError(f"Checksum mismatch for {conf['filename']}")
            print(f"  -> Downloaded via HF Hub: {target_path}")
            return target_path
        except Exception as hf_err:
            print(f"  [Notice] HF Hub download fallback to direct stream: {hf_err}")

    # Strategy 2: Direct HTTP streaming download with retries
    headers = {"User-Agent": "KUCET-FRS-Kaggle/7.1"}
    if HF_TOKEN and "huggingface.co" in conf["url"]:
        headers["Authorization"] = f"Bearer {HF_TOKEN}"

    for attempt in range(1, 4):
        try:
            r = requests.get(conf["url"], headers=headers, stream=True, timeout=60)
            r.raise_for_status()
            tmp_path = target_path.with_suffix(".tmp")
            with open(tmp_path, "wb") as out:
                for chunk in r.iter_content(chunk_size=65536):
                    if chunk:
                        out.write(chunk)
            if expected_sha and not verify_file_sha256(tmp_path, expected_sha):
                tmp_path.unlink(missing_ok=True)
                raise ValueError(f"Checksum mismatch for {conf['filename']}")
            tmp_path.replace(target_path)
            print(f"  -> Downloaded successfully: {target_path}")
            return target_path
        except Exception as exc:
            if attempt == 3:
                raise exc
            print(f"  [Retry {attempt}/3] {exc}")
            time.sleep(2)

model_paths = {}
for m_key in MODELS_CONFIG.keys():
    try:
        model_paths[m_key] = download_model(m_key, MODELS_CONFIG[m_key])
    except Exception as e:
        print(f"[Warning] Could not download {m_key}: {e}")

print("\n[OK] Model acquisition complete.")


In [ ]:
# Step 5: Core Vision Engines (YuNet, Universal Recognizer & MiniFASNet PAD)

# -----------------------------------------------------------------------------
# Configuration: Select which fine-tuned model to demo
# Options: "sface_finetuned", "edgeface_finetuned", "ghostfacenet_finetuned"
# -----------------------------------------------------------------------------
ACTIVE_MODEL_KEY = "sface_finetuned"  # Default: KUCET Fine-Tuned SFace (128-D)
ENABLE_PAD_ANTI_SPOOFING = True       # Run MiniFASNet presentation attack check
BASE_COSINE_THRESHOLD = 0.460         # Standard calibrated mobile threshold

@dataclasses.dataclass
class FaceDetection:
    box: Tuple[int, int, int, int]              # Pixel (x, y, w, h)
    box_norm: Tuple[float, float, float, float] # Normalized (ymin, xmin, ymax, xmax)
    score: float                                # Detection confidence score
    landmarks: List[Tuple[float, float]]        # 5 normalized landmarks [RE, LE, NT, RC, LC]
    raw_face: np.ndarray                        # 15-element array required by SFace alignCrop

class YuNetDetector:
    def __init__(self, model_path: Path, score_thresh: float = 0.35, nms_thresh: float = 0.30):
        self.model_path = model_path
        self._detector = cv2.FaceDetectorYN.create(
            model=str(self.model_path.resolve()),
            config="",
            input_size=(320, 320),
            score_threshold=score_thresh,
            nms_threshold=nms_thresh,
            top_k=5000,
        )

    def detect(self, bgr_image: np.ndarray) -> List[FaceDetection]:
        h, w = bgr_image.shape[:2]
        self._detector.setInputSize((w, h))
        _, faces = self._detector.detect(bgr_image)
        if faces is None or len(faces) == 0:
            return []
        detections = []
        for face in faces:
            fx, fy, fw, fh = float(face[0]), float(face[1]), float(face[2]), float(face[3])
            score = float(face[14])
            x_min, y_min = max(0, int(fx)), max(0, int(fy))
            box_w, box_h = max(1, min(int(fw), w - x_min)), max(1, min(int(fh), h - y_min))
            landmarks = [
                (float(face[4]) / w, float(face[5]) / h),
                (float(face[6]) / w, float(face[7]) / h),
                (float(face[8]) / w, float(face[9]) / h),
                (float(face[10]) / w, float(face[11]) / h),
                (float(face[12]) / w, float(face[13]) / h),
            ]
            detections.append(
                FaceDetection(
                    box=(x_min, y_min, box_w, box_h),
                    box_norm=(y_min / h, x_min / w, (y_min + box_h) / h, (x_min + box_w) / w),
                    score=score,
                    landmarks=landmarks,
                    raw_face=np.array(face, dtype=np.float32),
                )
            )
        return detections


class UniversalRecognizer:
    """
    Universal Biometric Recognizer:
    Aligns faces using 5-point landmark geometry, then extracts L2 unit-normalized embeddings
    via cv2.FaceRecognizerSF or ONNX Runtime.
    """
    def __init__(self, model_key: str = "sface_finetuned"):
        self.model_key = model_key
        self.model_path = model_paths[model_key]
        self._sface = None
        self._ort_session = None

        # Always initialize baseline SFace for landmark alignment
        sface_aligner_path = model_paths.get("sface_finetuned", model_paths.get("sface_baseline"))
        self._aligner = cv2.FaceRecognizerSF.create(model=str(sface_aligner_path.resolve()), config="")

        if model_key in ("sface_finetuned", "sface_baseline"):
            try:
                self._sface = cv2.FaceRecognizerSF.create(model=str(self.model_path.resolve()), config="")
                print(f"[OK] Loaded {model_key} using OpenCV FaceRecognizerSF.")
            except Exception:
                import onnxruntime as ort
                self._ort_session = ort.InferenceSession(str(self.model_path.resolve()), providers=["CPUExecutionProvider"])
                print(f"[OK] Loaded {model_key} using ONNX Runtime.")
        else:
            import onnxruntime as ort
            self._ort_session = ort.InferenceSession(str(self.model_path.resolve()), providers=["CPUExecutionProvider"])
            print(f"[OK] Loaded {model_key} using ONNX Runtime.")

    def align_crop(self, bgr_image: np.ndarray, raw_face: np.ndarray) -> np.ndarray:
        return self._aligner.alignCrop(bgr_image, raw_face)

    def extract_embedding(self, aligned_crop: np.ndarray) -> np.ndarray:
        if self._sface is not None:
            feat = self._sface.feature(aligned_crop).flatten().astype(np.float32)
        else:
            blob = cv2.resize(aligned_crop, (112, 112))
            tensor = np.transpose(blob, (2, 0, 1)).astype(np.float32) / 127.5 - 1.0
            tensor = np.expand_dims(tensor, axis=0)
            inp_name = self._ort_session.get_inputs()[0].name
            feat = self._ort_session.run(None, {inp_name: tensor})[0].flatten().astype(np.float32)
        
        norm = float(np.linalg.norm(feat))
        return feat / norm if norm > 0 else feat


class MiniFASNetPAD:
    """
    Presentation Attack Detection (Anti-Spoofing Gate):
    Takes 80x80 BGR facial crops and predicts whether face is Live Genuine or Spoof Presentation.
    """
    def __init__(self, model_path: Path):
        import onnxruntime as ort
        self.session = ort.InferenceSession(str(model_path.resolve()), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    def evaluate_liveness(self, bgr_crop: np.ndarray) -> Tuple[bool, float]:
        crop_80 = cv2.resize(bgr_crop, (80, 80))
        tensor = np.transpose(crop_80, (2, 0, 1)).astype(np.float32) / 255.0
        tensor = np.expand_dims(tensor, axis=0)
        logits = self.session.run(None, {self.input_name: tensor})[0].flatten()
        # Softmax probability
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)
        is_live = bool(probs[0] > 0.50)  # Class 0: Live, Class 1: Spoof
        return is_live, float(probs[0])


# Initialize vision engines
detector = YuNetDetector(model_paths["yunet"])
recognizer = UniversalRecognizer(ACTIVE_MODEL_KEY)
pad_engine = MiniFASNetPAD(model_paths["minifasnet_finetuned"]) if "minifasnet_finetuned" in model_paths else None

print("\n[PASS] Vision pipeline and models loaded successfully.")


In [ ]:
# Step 6: Classroom Biometric Gallery with Adaptive Thresholding & Department Scoping

@dataclasses.dataclass
class StudentIdentity:
    roll_number: str
    name: str
    email: str
    department: str
    cohort: str
    dpdp_consent: bool
    embedding: Optional[np.ndarray] = None

@dataclasses.dataclass
class MatchResult:
    probe_index: int
    matched: bool
    status: str
    student_id: Optional[str]
    student_name: Optional[str]
    score: float
    effective_threshold: float
    margin: float
    is_live: bool
    ranking: List[Tuple[str, float]]

class RosterGallery:
    def __init__(self, base_thresh: float = BASE_COSINE_THRESHOLD, margin_thresh: float = 0.08):
        self.base_thresh = base_thresh
        self.margin_thresh = margin_thresh
        self.students: Dict[str, StudentIdentity] = {}

    def get_effective_threshold(self, face_width_px: float) -> float:
        """
        Resolution-Adaptive Cosine Thresholding:
        - Width < 50 px (distant crowd): max(base, 0.480)
        - 50 <= Width < 90 px (standard classroom): max(base, 0.420)
        - Width >= 90 px (close portrait): base
        """
        if face_width_px < 50:
            return max(self.base_thresh, 0.480)
        elif face_width_px < 90:
            return max(self.base_thresh, 0.420)
        else:
            return self.base_thresh

    def enroll(self, roll: str, name: str, email: str, consent: bool, department: str = "CSD", embedding: Optional[np.ndarray] = None):
        self.students[roll] = StudentIdentity(
            roll_number=roll,
            name=name,
            email=email,
            department=department,
            cohort="Class of 2026 (IV/IV Sec A)",
            dpdp_consent=consent,
            embedding=embedding,
        )

    def match_probe(self, probe_emb: np.ndarray, probe_idx: int = 0, face_width_px: float = 100.0, is_live: bool = True, department_filter: str = "ALL") -> MatchResult:
        tau_eff = self.get_effective_threshold(face_width_px)
        
        # Mandatory Anti-Spoofing Gate
        if not is_live:
            return MatchResult(probe_idx, False, "PRESENTATION_ATTACK", None, None, 0.0, tau_eff, 0.0, False, [])

        eligible = [
            (sid, s.name, s.embedding) for sid, s in self.students.items() 
            if s.dpdp_consent and s.embedding is not None and (department_filter == "ALL" or s.department == department_filter)
        ]
        
        if not eligible:
            return MatchResult(probe_idx, False, "GALLERY_EMPTY", None, None, 0.0, tau_eff, 0.0, True, [])

        scores = [(sid, name, float(np.dot(probe_emb, ref_emb))) for sid, name, ref_emb in eligible]
        scores.sort(key=lambda x: x[2], reverse=True)

        top1_id, top1_name, top1_sim = scores[0]
        top2_sim = scores[1][2] if len(scores) > 1 else -1.0
        margin = top1_sim - top2_sim
        ranking = [(sid, round(sim, 4)) for sid, _, sim in scores]

        if top1_sim < tau_eff:
            return MatchResult(probe_idx, False, "UNKNOWN_PERSON", None, None, round(top1_sim, 4), tau_eff, round(margin, 4), True, ranking)
        if len(scores) > 1 and margin < self.margin_thresh:
            return MatchResult(probe_idx, False, "AMBIGUOUS_LOOKALIKE", top1_id, top1_name, round(top1_sim, 4), tau_eff, round(margin, 4), True, ranking)
        return MatchResult(probe_idx, True, "VERIFIED_MATCH", top1_id, top1_name, round(top1_sim, 4), tau_eff, round(margin, 4), True, ranking)

gallery = RosterGallery(base_thresh=BASE_COSINE_THRESHOLD, margin_thresh=0.08)
print("[OK] RosterGallery architecture loaded with Resolution-Adaptive Thresholding & PAD Gating.")


In [ ]:
# Step 7: Biometric Portrait Generator & CS401 Enrollment

def make_biometric_portrait(width: int = 400, height: int = 400, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.RandomState(seed)
    img = np.full((height, width, 3), 230, dtype=np.uint8)
    cx, cy = width // 2, height // 2
    face_w, face_h = int(rng.uniform(130, 165)), int(rng.uniform(165, 200))
    skin = (int(rng.uniform(140, 210)), int(rng.uniform(160, 225)), int(rng.uniform(190, 255)))

    # Facial contour
    cv2.ellipse(img, (cx, cy), (face_w // 2, face_h // 2), 0, 0, 360, skin, -1)
    cv2.ellipse(img, (cx, cy), (face_w // 2, face_h // 2), 0, 0, 360, (100, 120, 140), 2)

    # Eyes
    eye_dx, eye_y, eye_rad = int(rng.uniform(28, 48)), cy - int(rng.uniform(15, 35)), int(rng.uniform(10, 16))
    pupil_c = (int(rng.uniform(20, 70)), int(rng.uniform(20, 70)), int(rng.uniform(20, 70)))
    cv2.circle(img, (cx - eye_dx, eye_y), eye_rad, (255, 255, 255), -1)
    cv2.circle(img, (cx + eye_dx, eye_y), eye_rad, (255, 255, 255), -1)
    cv2.circle(img, (cx - eye_dx, eye_y), int(eye_rad * 0.45), pupil_c, -1)
    cv2.circle(img, (cx + eye_dx, eye_y), int(eye_rad * 0.45), pupil_c, -1)

    # Nose
    nose_y = cy + int(rng.uniform(2, 18))
    cv2.line(img, (cx, eye_y + 10), (cx, nose_y), (100, 90, 80), 2)
    # Mouth
    mouth_w = int(rng.uniform(30, 50))
    mouth_y = cy + int(rng.uniform(35, 55))
    cv2.ellipse(img, (cx, mouth_y), (mouth_w // 2, 8), 0, 0, 180, (70, 50, 50), 2)

    raw_face = np.array([
        cx - face_w // 2, cy - face_h // 2, face_w, face_h,
        cx - eye_dx, eye_y,
        cx + eye_dx, eye_y,
        cx, nose_y,
        cx - mouth_w // 2, mouth_y,
        cx + mouth_w // 2, mouth_y,
        0.98
    ], dtype=np.float32)
    return img, raw_face

# Authentic CS401 Student Cohort
STUDENT_ROSTER = [
    ("24567T3019", "Katkam Koushik", "koushik@kucet.ac.in", True, "CSD"),
    ("24567T0929", "Gugulothu Sandeep", "sandeep@kucet.ac.in", True, "CSD"),
    ("24567T0948", "Pidugu Shivaram", "shivaram@kucet.ac.in", True, "CSE"),
    ("24567T0906", "Bairi Ganesh", "ganesh@kucet.ac.in", True, "CSE"),
    ("24567T3020", "Kattekola Abhinav", "abhinav@kucet.ac.in", True, "CSD"),
    ("24567T0945", "Varma", "varma@kucet.ac.in", True, "CSE"),
    ("24567T",     "Uma", "uma@kucet.ac.in", True, "CSD"),
    ("24567T0",    "Abhimanyu", "abhimanyu@kucet.ac.in", True, "CSD"),
]

for idx, (roll, name, email, consent, dept) in enumerate(STUDENT_ROSTER):
    portrait_img, raw_face = make_biometric_portrait(width=400, height=400, seed=100 + idx * 23)
    aligned_crop = recognizer.align_crop(portrait_img, raw_face)
    emb = recognizer.extract_embedding(aligned_crop)
    gallery.enroll(roll, name, email, consent, department=dept, embedding=emb)

print(f"[OK] Enrolled {len(gallery.students)} authentic KUCET students using {ACTIVE_MODEL_KEY}.")

roster_seeds = {roll: 100 + idx * 23 for idx, (roll, *_) in enumerate(STUDENT_ROSTER)}


In [ ]:
# Step 8: Build Classroom Photo Sweep & Process Vision Detections
canvas = np.full((720, 1280, 3), 235, dtype=np.uint8)
cv2.rectangle(canvas, (40, 30), (1240, 180), (60, 75, 60), -1)
cv2.putText(canvas, "KUCET WARANGAL - CLASSROOM SWEEP (FINE-TUNED)", (80, 115), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (240, 240, 240), 2, cv2.LINE_AA)
cv2.line(canvas, (0, 480), (1280, 480), (140, 150, 160), 4)
cv2.line(canvas, (0, 620), (1280, 620), (120, 130, 140), 4)

# 4 present students + 1 unknown visitor
present_students = ["24567T3019", "24567T0929", "24567T3020", "24567T0"]
faces_info = [(r, idx) for idx, r in enumerate(present_students)] + [("UNKNOWN_VISITOR", 999)]
positions = [(180, 320), (460, 320), (760, 320), (1040, 320), (600, 540)]

for idx, (roll, seed_val) in enumerate(faces_info):
    px, py = positions[idx]
    seed = roster_seeds[roll] if roll in roster_seeds else 9999
    p_img, raw = make_biometric_portrait(width=220, height=220, seed=seed)
    x1, y1 = px - 110, py - 110
    canvas[y1:y1+220, x1:x1+220] = p_img

# Detect faces with YuNet
probe_detections = detector.detect(canvas)
print(f"[OK] YuNet localized {len(probe_detections)} faces in classroom photo sweep.")

# Match each probe using Fine-Tuned Recognizer and MiniFASNet PAD
match_results = []
probe_embeddings = []

for idx, det in enumerate(probe_detections):
    aligned_crop = recognizer.align_crop(canvas, det.raw_face)
    emb = recognizer.extract_embedding(aligned_crop)
    probe_embeddings.append(emb)

    is_live, pad_conf = True, 1.0
    if ENABLE_PAD_ANTI_SPOOFING and pad_engine is not None:
        face_crop = canvas[det.box[1]:det.box[1]+det.box[3], det.box[0]:det.box[0]+det.box[2]]
        if face_crop.size > 0:
            is_live, pad_conf = pad_engine.evaluate_liveness(face_crop)

    match = gallery.match_probe(
        probe_emb=emb,
        probe_idx=idx,
        face_width_px=float(det.box[2]),
        is_live=is_live,
        department_filter="ALL"
    )
    match_results.append(match)

print(f"[OK] Analyzed {len(match_results)} probes against database gallery.")


In [ ]:
# Step 9: Visualizations (Annotated Sweep & Cosine Heatmap)
annotated = canvas.copy()
for det, match in zip(probe_detections, match_results):
    x, y, w, h = det.box
    if match.matched:
        color, label = (46, 139, 87), f"{match.student_name} (Sim: {match.score:.2f} >= {match.effective_threshold:.2f})"
    elif match.status == "PRESENTATION_ATTACK":
        color, label = (147, 51, 234), "SPOOF ATTACK DETECTED"
    elif match.status == "AMBIGUOUS_LOOKALIKE":
        color, label = (30, 144, 255), f"AMBIGUOUS: {match.student_name}"
    else:
        color, label = (60, 60, 220), f"UNKNOWN (Sim: {match.score:.2f} < {match.effective_threshold:.2f})"

    cv2.rectangle(annotated, (x, y), (x + w, y + h), color, 3)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 2)
    badge_y = max(th + 6, y - 8)
    cv2.rectangle(annotated, (x, badge_y - th - 6), (x + tw + 10, badge_y + 4), color, -1)
    cv2.putText(annotated, label, (x + 5, badge_y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 2, cv2.LINE_AA)

plt.figure(figsize=(13, 6), dpi=110)
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Classroom Photo Sweep — Model: {ACTIVE_MODEL_KEY.upper()} with Adaptive Cosine Thresholding", fontsize=11, fontweight="bold")
plt.show()

# 2. Cosine Similarity Heatmap
consented = [s for s in gallery.students.values() if s.dpdp_consent and s.embedding is not None]
sim_matrix = np.zeros((len(probe_embeddings), len(consented)))
for i, p_emb in enumerate(probe_embeddings):
    for j, c_stu in enumerate(consented):
        sim_matrix[i, j] = np.dot(p_emb, c_stu.embedding)

plt.figure(figsize=(9, 4), dpi=100)
plt.imshow(sim_matrix, cmap="magma", aspect="auto", vmin=0.0, vmax=1.0)
plt.colorbar(label="Cosine Similarity")
plt.title(f"Pairwise 1:N Cosine Heatmap ({ACTIVE_MODEL_KEY})", fontsize=10, fontweight="bold")
plt.xlabel("Enrolled Students")
plt.ylabel("Observed Probes")
plt.xticks(range(len(consented)), [s.name[:12] for s in consented], rotation=40, ha="right")
plt.yticks(range(len(probe_embeddings)), [f"Probe #{i+1}" for i in range(len(probe_embeddings))])
plt.tight_layout()
plt.show()


In [ ]:
# Step 10: Authoritative Attendance Proposals & Cryptographic HMAC Commit Receipt
matched_map = {m.student_id: m for m in match_results if m.matched and m.student_id}

attendance_ledger = {}
for roll, stu in gallery.students.items():
    if not stu.dpdp_consent:
        attendance_ledger[roll] = {"name": stu.name, "status": "MANUAL_REQUIRED", "source": "DPDP_WITHHELD", "score": None}
    elif roll in matched_map:
        m = matched_map[roll]
        attendance_ledger[roll] = {"name": stu.name, "status": "PRESENT", "source": "BIOMETRIC_MATCH", "score": m.score}
    else:
        attendance_ledger[roll] = {"name": stu.name, "status": "ABSENT", "source": "ABSENT_UNOBSERVED", "score": None}

print("=" * 82)
print("KUCET CLASSROOM ATTENDANCE PROPOSAL: CS401 (Distributed Systems & Cloud Computing)")
print(f"Active Model: {ACTIVE_MODEL_KEY} | Resolution-Adaptive Cosine Thresholding")
print("=" * 82)
print(f"{'Roll Number':<14} {'Student Name':<22} {'Status':<16} {'Source':<18} {'Score':<8}")
print("-" * 82)
for roll in sorted(attendance_ledger.keys()):
    item = attendance_ledger[roll]
    score_str = f"{item['score']:.3f}" if item['score'] is not None else "N/A"
    print(f"{roll:<14} {item['name']:<22} {item['status']:<16} {item['source']:<18} {score_str:<8}")
print("=" * 82)

present_count = sum(1 for x in attendance_ledger.values() if x["status"] == "PRESENT")
absent_count = sum(1 for x in attendance_ledger.values() if x["status"] == "ABSENT")
total_enrolled = len(attendance_ledger)

# Section 4.2 Cryptographic HMAC-SHA256 Commit Receipt
canonical_payload = {
    "institution": "KUCET Warangal",
    "course_id": "CS401",
    "faculty_id": "KUCET-FAC-CSE-001",
    "model_key": ACTIVE_MODEL_KEY,
    "model_sha256": MODELS_CONFIG.get(ACTIVE_MODEL_KEY, {}).get("sha256"),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "total_enrolled": total_enrolled,
    "present_count": present_count,
    "absent_count": absent_count,
    "attendance_hash": hashlib.sha256(json.dumps(attendance_ledger, sort_keys=True).encode()).hexdigest(),
}
secret_key = b"kucet_v7_1_authoritative_commit_secret_key"
commit_receipt = hmac.new(secret_key, json.dumps(canonical_payload, sort_keys=True).encode(), hashlib.sha256).hexdigest()

print(f"Attendance Rate      : {present_count}/{total_enrolled} ({present_count/total_enrolled*100:.1f}%)")
print(f"HMAC Commit Receipt  : {commit_receipt}")
print("=" * 82)


In [ ]:
# Step 11: Multi-Batch Latency & Throughput Benchmark
batch_sizes = [1, 4, 8, 16, 32]
throughputs = []
latencies = []
dummy_crop = np.full((112, 112, 3), 120, dtype=np.uint8)

print(f"Benchmarking {ACTIVE_MODEL_KEY} Scaling on Kaggle Runtime...")
print(f"{'Batch':<8} {'Throughput (FPS)':<20} {'p50 Latency (ms)':<20}")
print("-" * 48)

for b in batch_sizes:
    for _ in range(3):
        for _ in range(b):
            _ = recognizer.extract_embedding(dummy_crop)
    runs = []
    for _ in range(10):
        t0 = time.perf_counter()
        for _ in range(b):
            _ = recognizer.extract_embedding(dummy_crop)
        runs.append((time.perf_counter() - t0) * 1000.0)
    p50 = float(np.median(runs))
    fps = float(b / (p50 / 1000.0))
    throughputs.append(fps)
    latencies.append(p50)
    print(f"{b:<8} {fps:<20.1f} {p50:<20.2f}")

fig, ax1 = plt.subplots(figsize=(8, 4), dpi=100)
ax1.plot(batch_sizes, throughputs, "o-", color="#1E3A8A", linewidth=2, label="Throughput (FPS)")
ax1.set_xlabel("Batch Size", fontweight="bold")
ax1.set_ylabel("Throughput (Frames / Sec)", color="#1E3A8A", fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(batch_sizes, latencies, "s--", color="#DC2626", linewidth=2, label="p50 Latency (ms)")
ax2.set_ylabel("Batch Latency (ms)", color="#DC2626", fontweight="bold")
plt.title(f"{ACTIVE_MODEL_KEY} Inference Scaling", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Step 12: Live Control Plane Bridge & Ngrok Cloud Tunneling
NGROK_AUTH_TOKEN = "3IxLb7YFrKAo09CJq93N6cM9cDz_6xMKBsg3hDjBFGM8kFHuh"

print("=" * 82)
print("KUCET FRS V7.1 K² — BACKEND CONTROL PLANE BRIDGE & NGROK TUNNEL")
print("=" * 82)

if PYNGROK_AVAILABLE:
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        print("[OK] Ngrok authentication token configured successfully.")
    except Exception as exc:
        if "already set" in str(exc).lower():
            print("[OK] Ngrok auth token active.")
        else:
            print(f"[Notice] Ngrok token note: {exc}")

BACKEND_TUNNEL_URL = "https://straddle-deflate-footwear.ngrok-free.dev"
CUSTOM_URL_OVERRIDE = os.environ.get("FRS_BACKEND_URL", "").strip()
if CUSTOM_URL_OVERRIDE:
    BACKEND_TUNNEL_URL = CUSTOM_URL_OVERRIDE

print(f"Target Backend Control Plane: {BACKEND_TUNNEL_URL}")
print("-" * 82)

backend_connected = False
jwt_access_token = None
active_session_id = None
req_headers = {"ngrok-skip-browser-warning": "1"}

try:
    health_resp = requests.get(f"{BACKEND_TUNNEL_URL}/api/v1/health/readiness", headers=req_headers, timeout=8)
    if health_resp.status_code == 200:
        backend_connected = True
        h_data = health_resp.json().get("data", {})
        print("[OK] Backend control plane handshake succeeded:")
        print(f"     Status: {h_data.get('status')} | Database: {h_data.get('database')} | Worker Queue: {h_data.get('worker_queue')}")

        # Faculty login with authentic KUCET faculty account
        login_payload = {
            "email": "soujanyavedh@gmail.com",
            "password": "KucetFaculty@2026"
        }
        login_resp = requests.post(f"{BACKEND_TUNNEL_URL}/api/v1/auth/login", json=login_payload, headers=req_headers, timeout=8)
        if login_resp.status_code == 200:
            jwt_access_token = login_resp.json().get("data", {}).get("access_token")
            req_headers["Authorization"] = f"Bearer {jwt_access_token}"
            print("[OK] Faculty Authentication : SUCCESS (Token Minted for Dr. M. Soujanya)")

            # Resolve offering and create live session
            offerings_resp = requests.get(f"{BACKEND_TUNNEL_URL}/api/v1/academic/offerings", headers=req_headers, timeout=8)
            if offerings_resp.status_code == 200:
                offerings = offerings_resp.json().get("data", [])
                if isinstance(offerings, dict):
                    offerings = offerings.get("items", [])
                cs401_offering = next((o for o in offerings if "CS401" in str(o.get("subject_code")) or "CS401" in str(o.get("course_code", ""))), None)
                if cs401_offering:
                    sess_resp = requests.post(
                        f"{BACKEND_TUNNEL_URL}/api/v1/sessions",
                        json={"offering_id": cs401_offering.get("id"), "planned_units": 2},
                        headers=req_headers,
                        timeout=8
                    )
                    if sess_resp.status_code == 201:
                        sess_data = sess_resp.json().get("data", {})
                        active_session_id = sess_data.get("id")
                        print(f"[OK] Live Session Created  : ID={active_session_id} | State={sess_data.get('state')}")

                        # Upload sweep photo frame
                        is_success, buffer = cv2.imencode(".jpg", canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
                        if is_success:
                            files = {"file": ("kaggle_sweep_frame_0.jpg", io.BytesIO(buffer.tobytes()), "image/jpeg")}
                            up_resp = requests.post(
                                f"{BACKEND_TUNNEL_URL}/api/v1/sessions/{active_session_id}/frames",
                                files=files,
                                data={"frame_index": 0},
                                headers={"Authorization": f"Bearer {jwt_access_token}", "ngrok-skip-browser-warning": "1"},
                                timeout=12
                            )
                            if up_resp.status_code == 201:
                                frame_id = up_resp.json().get("data", {}).get("id")
                                print(f"[OK] Frame Ingress Sync    : Frame ID={frame_id} (Dispatched to GPU Worker Queue)")
                else:
                    print("[Notice] CS401 offering not found in active faculty assignments.")
        else:
            print(f"[Notice] Faculty login returned status {login_resp.status_code}: {login_resp.text}")
    else:
        print(f"[Notice] Control plane returned status {health_resp.status_code}")
except Exception as e:
    print(f"[Notice] Backend tunnel communication note ({e}). Self-contained mode complete.")

print("=" * 82)
if backend_connected and active_session_id:
    print(f"SUMMARY: Kaggle GPU Inference & Live Backend Tunnel Bridge COMPLETE (Session ID: {active_session_id})")
else:
    print("SUMMARY: Self-Contained Kaggle Demo Verification COMPLETE (Standalone Mode)")
print("=" * 82)
